In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import scienceplots

from FastBEMT import (
    BEMT,
    Environment,
    Propeller,
    Simulation,
    load_propeller_geometry,
)

plt.style.use(["science", "no-latex"])


In [ ]:
geometry = load_propeller_geometry("Data/10x7E.pkl")
environment = Environment()
simulation = Simulation(
    revolutions=1,
    timesteps_per_revolution=100,
    device="cpu",
)
propeller = Propeller(geometry, environment, simulation)


In [ ]:
rpm = 7000.0
advance_ratios = np.arange(0.0, 1.0, 0.1)

bemt = BEMT(
    propeller=propeller,
    rpm=rpm,
    J=advance_ratios,
)
performance = bemt.performance.reset_index()
diameter = 2.0 * geometry["tip_radius"]
performance["advance_ratio"] = performance["v_inf"] / (
    performance["rpm"] / 60.0 * diameter
)
performance["efficiency"] = np.divide(
    performance["advance_ratio"] * performance["thrust_coefficient"],
    2.0 * np.pi * performance["torque_coefficient"],
    out=np.zeros(len(performance)),
    where=performance["torque_coefficient"].to_numpy() > 0.0,
)
performance


In [ ]:
figure, axes = plt.subplots(1, 3, figsize=(15, 4))

plot_columns = (
    ("thrust_coefficient", "Thrust coefficient, $C_T$", (0.0, 0.15)),
    ("torque_coefficient", "Torque coefficient, $C_Q$", (0.0, 0.08 / (2.0 * np.pi))),
    ("efficiency", r"Propulsive efficiency, $\eta$", (0.0, 0.8)),
)
for axis, (column, ylabel, limits) in zip(axes, plot_columns):
    axis.plot(
        performance["advance_ratio"],
        performance[column],
        "o-",
        linewidth=2,
        markersize=4,
    )
    axis.set_xlabel("Advance ratio, J")
    axis.set_ylabel(ylabel)
    axis.set_ylim(*limits)
    axis.grid(True, linestyle=":")
    axis.spines[["top", "right"]].set_visible(False)
    axis.tick_params(top=False, right=False, which="both")

figure.tight_layout()
figure_path = Path("../Figures/Validation/bemt_performance.pdf")
figure_path.parent.mkdir(parents=True, exist_ok=True)
figure.savefig(figure_path, dpi=200)
plt.show()
